# Lab 01 · Digital Subtraction Angiography
**Lý Chính Tân · HCMUT · Medical Imaging coursework**

This notebook follows the supplied lab sequence: load mask/live images, perform signed subtraction,
compare contrast processing and Gaussian smoothing, then register a rotated live image before subtraction.
The final section preserves the original exploratory black-hat, CLAHE and Frangi enhancements.

**Run order:** setup → inputs → basic subtraction → smoothing → registration → enhancement → exports.
The default uses explicitly synthetic images. Select `MODE='local'` for the supplied lab data.

## 1. Configuration and dependencies

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'imaging_labs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the medical-imaging-labs repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from imaging_labs.common import normalize_display, image_metrics, show_images, save_figure

# Choose 'demo' or 'local'. Local mode never falls back silently to demo data.
MODE = os.environ.get('IMAGING_LABS_MODE', 'demo')
DATA_ROOT = Path(os.environ.get('IMAGING_LABS_DATA', str(ROOT / 'data')))
OUTPUT_ROOT = Path(os.environ.get('IMAGING_LABS_OUTPUT', str(ROOT / 'results'))) / MODE
if MODE not in {'demo', 'local'}:
    raise ValueError("MODE must be 'demo' or 'local'")
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})
print('Input mode:', MODE)

In [ ]:
from imaging_labs.dsa import load_grayscale, subtract_images, register_live, enhance_vessels
from imaging_labs.demo import dsa_demo
import cv2

DSA_DIR = DATA_ROOT / 'dsa'
OUTPUT = OUTPUT_ROOT / 'dsa'
OUTPUT.mkdir(parents=True, exist_ok=True)
GAUSSIAN_SIGMA = 1.0
REGISTRATION_MODEL = 'affine'

## 2. Load the mask, live image and rotated live image
The two images must have the same grid. Resizing is not performed automatically because that changes
the subtraction problem. Pixel values are converted to floating point before arithmetic.

In [ ]:
if MODE == 'demo':
    mask, live, rotated_live = dsa_demo()
else:
    mask = load_grayscale(DSA_DIR / 'mask.jpg')
    live = load_grayscale(DSA_DIR / 'live.jpg')
    rotated_live = load_grayscale(DSA_DIR / 'rotated_live.jpg')
if not mask.shape == live.shape == rotated_live.shape:
    raise ValueError('DSA inputs must share an image grid; check the selected files.')
print('Image shape:', mask.shape)
fig = show_images([mask, live, rotated_live], ['Mask', 'Live', 'Rotated live'])
save_figure(fig, OUTPUT, '01_inputs.png')
plt.show()

## 3. Signed subtraction and contrast display
Compute `live - mask` in floating point to preserve negative values. Absolute subtraction is a separate
display choice, not an interchangeable numerical result. Apply histogram equalization **after**
subtraction: separate nonlinear equalization of mask and live would change their intensity relationship.

In [ ]:
signed = subtract_images(mask, live)
sub_u8 = np.round(255 * normalize_display(signed, (0, 100))).astype('uint8')
equalized = cv2.equalizeHist(sub_u8)
fig = show_images([signed, sub_u8, equalized],
                  ['Signed live - mask', 'Min-max display', 'Histogram equalization'])
save_figure(fig, OUTPUT, '02_subtraction.png')
plt.show()
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(sub_u8.ravel(), bins=64, alpha=.6, label='Min-max')
ax.hist(equalized.ravel(), bins=64, alpha=.6, label='Equalized')
ax.set(xlabel='Display intensity', ylabel='Pixel count', title='Contrast redistribution')
ax.legend()
plt.show()

## 4. Gaussian smoothing before and after subtraction
With the same Gaussian operator and boundary handling, linearity makes the two paths approximately
equal. Smoothing suppresses noise but can blur small vessels; it is not proof of improved diagnostic quality.

In [ ]:
pre_smoothed = subtract_images(mask, live, sigma=GAUSSIAN_SIGMA)
post_smoothed = cv2.GaussianBlur(signed, (0, 0), GAUSSIAN_SIGMA)
print('Maximum linearity difference:', float(np.max(np.abs(pre_smoothed-post_smoothed))))
fig = show_images([signed, pre_smoothed, post_smoothed],
                  ['No smoothing', 'Smooth mask/live first', 'Smooth subtraction afterward'])
save_figure(fig, OUTPUT, '03_smoothing.png')
plt.show()

## 5. Registration before subtraction
A coarse-to-fine NCC rotation search initializes alignment. Phase correlation estimates translation;
ECC then refines the requested transform. Warped border pixels are excluded from statistics.
Registration similarity can be affected by contrast injection and must be checked visually.

In [ ]:
registration = register_live(mask, rotated_live, motion=REGISTRATION_MODEL)
registered = registration['registered']
valid = registration['valid']
registered_subtraction = subtract_images(mask, registered)
unregistered_subtraction = subtract_images(mask, rotated_live)
summary = {
    'input_mode': MODE,
    'rotation_deg': registration['rotation_deg'],
    'ecc_correlation': registration['ecc_correlation'],
    'valid_fraction': float(valid.mean()),
    'mean_absolute_subtraction_before': float(np.mean(np.abs(unregistered_subtraction[valid]))),
    'mean_absolute_subtraction_after': float(np.mean(np.abs(registered_subtraction[valid]))),
}
display(pd.DataFrame([summary]))
signed_display = np.where(valid, registered_subtraction, np.nan)
fig = show_images([rotated_live, registered, unregistered_subtraction, signed_display, valid],
                  ['Rotated live', 'Registered live', 'Before registration',
                   'After registration (valid area)', 'Valid overlap'])
save_figure(fig, OUTPUT, '04_registration.png')
plt.show()

## 6. Exploratory vessel enhancement
These variants enhance dark ridges in the **unrotated** subtraction image. They do not produce
validated vessel segmentation. The Frangi implementation uses the current `sigmas` argument and
is labeled accurately; failure no longer silently substitutes a different method.

In [ ]:
enhancements = enhance_vessels(signed)
fig = show_images(list(enhancements.values()), list(enhancements))
save_figure(fig, OUTPUT, '05_vessel_enhancement.png')
plt.show()

## 7. Export and interpretation

In [ ]:
np.savez_compressed(OUTPUT / 'subtraction_arrays.npz', signed=signed,
                    registered=registered, registered_subtraction=registered_subtraction,
                    valid_overlap=valid)
(OUTPUT / 'registration_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved figures, signed arrays and registration summary in the selected output folder.')

### Discussion
- Signed subtraction preserves the relationship between mask and contrast-enhanced images.
- Histogram equalization changes display contrast; it does not restore lost information.
- Registration can reduce motion residuals, but residual magnitude also contains true vessel signal.
- All enhancement parameters are educational settings, not clinically validated thresholds.

**References:** supplied Lab 1 DSA handout, Le Nhat Tan, HCMUT;
[OpenCV ECC](https://docs.opencv.org/4.x/dc/d6b/group__video__track.html);
[scikit-image filters](https://scikit-image.org/docs/stable/api/skimage.filters.html).